# 04 · Görsel model (Adım 8, kutu 3) — ChangeOS pilot
Kaggle: **GPU T4 x2**, internet açık, dataset `melihkotman/disasterm3-turkiye` ve (GitHub planı bitene kadar) kod için `kod` dataset'i ekli.
Protokol ve karar kuralları: `docs/adim8_pilot_kaggle.md` (sonuçlardan ÖNCE yazıldı). **Test bölmesine ve Osmaniye katlamasına dokunulmaz.**

In [ ]:
# 1) Kurulum (ever-beta 0.5.6 sabit!)
!pip install -q timm "ever-beta==0.5.6" albumentations tifffile einops "datasets[vision]"
!pip install -q --no-deps "git+https://github.com/Z-Zheng/pytorch-change-models.git"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Kodu al: A) GitHub (plan yenilendiyse)   B) 'kod' dataset'i (aşağıda yolu uyarla)
import os, subprocess, sys, glob
REPO = "/kaggle/working/deprem-vlm-rag"
if not os.path.exists(REPO):
    cand = glob.glob("/kaggle/input/**/deprem-vlm-rag", recursive=True)
    if cand:
        subprocess.run(["cp", "-r", cand[0], REPO], check=True)
    else:
        # GitHub özel repo: Secrets'ta GITHUB_TOKEN 'Attached' olmalı
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        subprocess.run(["git", "clone", f"https://{tok}@github.com/MelihKotman/Deprem-VLM-RAG.git", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/MelihKotman/Deprem-VLM-RAG.git"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
!python -m src.config && python -m src.dm3_verify

In [ ]:
# 3) Birim testleri (model/GPU gerekmez; 1 dk): ölçüt ve oylama kodu doğru mu?
!python -m src.selftest_eval

## Aşama 0 — hat testi (≈10–15 dk): süre, bellek, veri bekleme, kayıp düşüyor mu
`--pilot`: 64 eğitim karosu, 3 epoch, 40 karoluk değerlendirme. Değerlendirme **Malatya katlaması** (Adıyaman+Osmaniye ile eğitilir). Çıktıdaki `EXTRAPOLATION...` tahmindir, ölçüm değildir.

In [ ]:
!python -m src.train_changeos --scenario S1 --holdout Malatya --pilot --pilot-tiles 64 --epochs 3 \
    --post-shift 16 --cache-dir /kaggle/working/cache --workers 4 --out /kaggle/working/runs/p0_hat_testi

In [ ]:
!cat /kaggle/working/runs/p0_hat_testi/timing.json
!tail -n 3 /kaggle/working/runs/p0_hat_testi/log.jsonl

## Aşama 1 — sonrası öteleme aralığı (0 / 16 / 32 px), diğer her şey sabit
Kota/süre Aşama 0'daki ölçüme göre `--epochs` ile ayarlanır (öneri: 12). Üç koşu aynı seed (42). Her biri `--max-minutes` ile korunur; kesilirse aynı komut + `--resume`.

In [ ]:
for R in (0, 16, 32):
    !python -m src.train_changeos --scenario S1 --holdout Malatya --epochs 12 --post-shift {R} \
        --cache-dir /kaggle/working/cache --workers 4 --max-minutes 60 --out /kaggle/working/runs/p1_shift{R}

## Aşama 2 — sınıf dengesi (en iyi öteleme sabit): hasar odaklı örnekleme ± CE sınıf ağırlığı
`--dam-p 0.5` (varsayılan) ile `--dam-p 0.5 --cls-weights 1,1,3,3` karşılaştırılır; ayrıca `--dam-p 0.0` (tek örnekleme, referans).

In [ ]:
BEST = 16   # Aşama 1 sonucuna göre elle gir
!python -m src.train_changeos --scenario S1 --holdout Malatya --epochs 12 --post-shift {BEST} --dam-p 0.0 \
    --cache-dir /kaggle/working/cache --workers 4 --max-minutes 60 --out /kaggle/working/runs/p2_dam0
!python -m src.train_changeos --scenario S1 --holdout Malatya --epochs 12 --post-shift {BEST} --dam-p 0.5 --cls-weights 1,1,3,3 \
    --cache-dir /kaggle/working/cache --workers 4 --max-minutes 60 --out /kaggle/working/runs/p2_w1133

In [ ]:
# Özet tablo (tüm koşular)
!python -m src.summarize_runs /kaggle/working/runs

Sonuçları (özet tablo + `timing.json`) olduğu gibi yapıştır; `notlar/adim8-durum.md` ve checklist oradan güncellenir.